# TIR Super-Resolution + Colorization Model Training Notebook

This notebook starts model development after dataset preparation.

It assumes the dataset has already been created in repo-compatible format:

```text
dataset_current_repo_format/
  sr/
    train/tir_200m/*.npy
    train/tir_100m/*.npy
    val/tir_200m/*.npy
    val/tir_100m/*.npy
    test/tir_200m/*.npy
    test/tir_100m/*.npy

  colorization/
    train/tir_100m/*.npy
    train/rgb_100m/*.npy
    val/tir_100m/*.npy
    val/rgb_100m/*.npy
    test/tir_100m/*.npy
    test/rgb_100m/*.npy

  metadata/
    patch_metadata.csv
    scene_summary.csv
```

Model tasks:

```text
Task 1: Super Resolution
Input  = TIR 200m, shape 1×256×256
Target = TIR 100m, shape 1×512×512

Task 2: Colorization
Input  = TIR 100m, shape 1×256×256
Target = RGB 100m, shape 3×256×256
```

Recommended workflow:

1. Run all dataset/dataloader checks.
2. Train the SR baseline first.
3. Train the colorization baseline second.
4. Save checkpoints.
5. Run inference visualization.
6. Later improve architectures/losses.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import random
import math
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Dataset paths

In [ ]:
# Main Drive dataset path
DATASET_ROOT = "/content/drive/MyDrive/landsat_india_200/dataset_current_repo_format"

# If you are using a local copied version, change this path:
# DATASET_ROOT = "/content/landsat_india_200_work/dataset_current"

assert os.path.exists(DATASET_ROOT), f"Dataset path not found: {DATASET_ROOT}"
print("Using dataset:", DATASET_ROOT)

for task in ["sr", "colorization"]:
    print("\nTASK:", task)
    for split in ["train", "val", "test"]:
        if task == "sr":
            folders = ["tir_200m", "tir_100m"]
        else:
            folders = ["tir_100m", "rgb_100m"]
        for folder in folders:
            p = os.path.join(DATASET_ROOT, task, split, folder)
            n = len(glob.glob(os.path.join(p, "*.npy"))) if os.path.exists(p) else 0
            print(f"{split}/{folder}: {n}")

## 3. Dataset classes

In [ ]:
class SRDataset(Dataset):
    '''
    Super-resolution dataset.

    Input:
      TIR 200m patch, shape 256×256, Kelvin

    Target:
      TIR 100m patch, shape 512×512, Kelvin

    Output tensors:
      x: [1, 256, 256], normalized 0–1
      y: [1, 512, 512], normalized 0–1
    '''
    def __init__(self, root, split="train", augment=False):
        self.root = root
        self.split = split
        self.augment = augment

        self.x_dir = os.path.join(root, "sr", split, "tir_200m")
        self.y_dir = os.path.join(root, "sr", split, "tir_100m")

        self.files = sorted(glob.glob(os.path.join(self.x_dir, "*.npy")))
        assert len(self.files) > 0, f"No SR files found in {self.x_dir}"

        # Keep only files that have matching targets
        self.files = [
            f for f in self.files
            if os.path.exists(os.path.join(self.y_dir, os.path.basename(f)))
        ]

        print(f"SRDataset {split}: {len(self.files)} samples")

    def __len__(self):
        return len(self.files)

    @staticmethod
    def normalize_tir(x):
        # Kelvin approx 250–350 -> 0–1
        x = (x - 250.0) / 100.0
        return np.clip(x, 0.0, 1.0)

    def maybe_augment(self, x, y):
        # x shape: [256,256], y shape: [512,512]
        if not self.augment:
            return x, y

        if random.random() < 0.5:
            x = np.flip(x, axis=1).copy()
            y = np.flip(y, axis=1).copy()

        if random.random() < 0.5:
            x = np.flip(x, axis=0).copy()
            y = np.flip(y, axis=0).copy()

        k = random.randint(0, 3)
        if k:
            x = np.rot90(x, k, axes=(0, 1)).copy()
            y = np.rot90(y, k, axes=(0, 1)).copy()

        return x, y

    def __getitem__(self, idx):
        x_path = self.files[idx]
        name = os.path.basename(x_path)
        y_path = os.path.join(self.y_dir, name)

        x = np.load(x_path).astype(np.float32)
        y = np.load(y_path).astype(np.float32)

        x = self.normalize_tir(x)
        y = self.normalize_tir(y)

        x, y = self.maybe_augment(x, y)

        x = torch.from_numpy(x).unsqueeze(0)  # [1,256,256]
        y = torch.from_numpy(y).unsqueeze(0)  # [1,512,512]

        return x, y, name


class ColorizationDataset(Dataset):
    '''
    Colorization dataset.

    Input:
      TIR 100m patch, shape 256×256, Kelvin

    Target:
      RGB 100m patch, shape 3×256×256, reflectance 0–1

    Output tensors:
      x: [1, 256, 256], normalized 0–1
      y: [3, 256, 256], clipped 0–1
    '''
    def __init__(self, root, split="train", augment=False):
        self.root = root
        self.split = split
        self.augment = augment

        self.x_dir = os.path.join(root, "colorization", split, "tir_100m")
        self.y_dir = os.path.join(root, "colorization", split, "rgb_100m")

        self.files = sorted(glob.glob(os.path.join(self.x_dir, "*.npy")))
        assert len(self.files) > 0, f"No colorization files found in {self.x_dir}"

        self.files = [
            f for f in self.files
            if os.path.exists(os.path.join(self.y_dir, os.path.basename(f)))
        ]

        print(f"ColorizationDataset {split}: {len(self.files)} samples")

    def __len__(self):
        return len(self.files)

    @staticmethod
    def normalize_tir(x):
        x = (x - 250.0) / 100.0
        return np.clip(x, 0.0, 1.0)

    def maybe_augment(self, x, y):
        # x shape: [256,256], y shape: [3,256,256]
        if not self.augment:
            return x, y

        if random.random() < 0.5:
            x = np.flip(x, axis=1).copy()
            y = np.flip(y, axis=2).copy()

        if random.random() < 0.5:
            x = np.flip(x, axis=0).copy()
            y = np.flip(y, axis=1).copy()

        k = random.randint(0, 3)
        if k:
            x = np.rot90(x, k, axes=(0, 1)).copy()
            y = np.rot90(y, k, axes=(1, 2)).copy()

        return x, y

    def __getitem__(self, idx):
        x_path = self.files[idx]
        name = os.path.basename(x_path)
        y_path = os.path.join(self.y_dir, name)

        x = np.load(x_path).astype(np.float32)
        y = np.load(y_path).astype(np.float32)

        x = self.normalize_tir(x)
        y = np.clip(y, 0.0, 1.0)

        x, y = self.maybe_augment(x, y)

        x = torch.from_numpy(x).unsqueeze(0)  # [1,256,256]
        y = torch.from_numpy(y)               # [3,256,256]

        return x, y, name

## 4. Create dataloaders

In [ ]:
BATCH_SIZE = 4
NUM_WORKERS = 2

sr_train_ds = SRDataset(DATASET_ROOT, "train", augment=True)
sr_val_ds = SRDataset(DATASET_ROOT, "val", augment=False)
sr_test_ds = SRDataset(DATASET_ROOT, "test", augment=False)

color_train_ds = ColorizationDataset(DATASET_ROOT, "train", augment=True)
color_val_ds = ColorizationDataset(DATASET_ROOT, "val", augment=False)
color_test_ds = ColorizationDataset(DATASET_ROOT, "test", augment=False)

sr_train_loader = DataLoader(sr_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
sr_val_loader = DataLoader(sr_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

color_train_loader = DataLoader(color_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
color_val_loader = DataLoader(color_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

x_sr, y_sr, names_sr = next(iter(sr_train_loader))
x_col, y_col, names_col = next(iter(color_train_loader))

print("SR input:", x_sr.shape)
print("SR target:", y_sr.shape)
print("Color input:", x_col.shape)
print("Color target:", y_col.shape)

## 5. Visual sanity check

In [ ]:
def show_batch_sample():
    x_sr, y_sr, names_sr = next(iter(sr_train_loader))
    x_col, y_col, names_col = next(iter(color_train_loader))

    idx = 0

    fig, ax = plt.subplots(1, 4, figsize=(18, 5))

    ax[0].imshow(x_sr[idx, 0].numpy(), cmap="gray")
    ax[0].set_title("SR Input\nTIR 200m 256×256")
    ax[0].axis("off")

    ax[1].imshow(y_sr[idx, 0].numpy(), cmap="gray")
    ax[1].set_title("SR Target\nTIR 100m 512×512")
    ax[1].axis("off")

    ax[2].imshow(x_col[idx, 0].numpy(), cmap="gray")
    ax[2].set_title("Color Input\nTIR 100m 256×256")
    ax[2].axis("off")

    rgb = y_col[idx].permute(1, 2, 0).numpy()
    ax[3].imshow(np.clip(rgb, 0, 1))
    ax[3].set_title("Color Target\nRGB 100m 256×256")
    ax[3].axis("off")

    plt.tight_layout()
    plt.show()

show_batch_sample()

## 6. Metrics and utility functions

In [ ]:
def psnr_from_mse(mse, max_val=1.0):
    if mse <= 0:
        return 99.0
    return 20 * math.log10(max_val / math.sqrt(mse))


def tensor_psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target).item()
    return psnr_from_mse(mse, max_val=max_val)


def save_checkpoint(model, optimizer, epoch, path, extra=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    payload = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)
    print("Saved checkpoint:", path)


def load_checkpoint(model, optimizer, path, device=DEVICE):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    print("Loaded checkpoint:", path)
    return ckpt

## 7. Super-resolution baseline model

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.net(x)


class SRResNetLite(nn.Module):
    '''
    Simple 2x super-resolution model.

    Input:
      [B,1,256,256]

    Output:
      [B,1,512,512]
    '''
    def __init__(self, in_channels=1, base_channels=64, num_blocks=6):
        super().__init__()

        self.head = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 9, padding=4),
            nn.ReLU(inplace=True),
        )

        self.body = nn.Sequential(
            *[ResidualBlock(base_channels) for _ in range(num_blocks)]
        )

        self.upsample = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True),
        )

        self.tail = nn.Conv2d(base_channels, 1, 9, padding=4)

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        x = x + res
        x = self.upsample(x)
        x = self.tail(x)
        return torch.sigmoid(x)


sr_model = SRResNetLite().to(DEVICE)

dummy = torch.randn(2, 1, 256, 256).to(DEVICE)
with torch.no_grad():
    out = sr_model(dummy)
print("SR output shape:", out.shape)

## 8. Train super-resolution model

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/landsat_india_200/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def train_sr_model(model, train_loader, val_loader, epochs=5, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for x, y, _ in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            pred = model(x)
            loss = criterion(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        val_psnr = 0.0

        with torch.no_grad():
            for x, y, _ in val_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)

                pred = model(x)
                loss = criterion(pred, y)

                val_loss += loss.item() * x.size(0)
                val_psnr += tensor_psnr(pred, y) * x.size(0)

        val_loss /= len(val_loader.dataset)
        val_psnr /= len(val_loader.dataset)

        print(f"Epoch {epoch:03d} | train L1: {train_loss:.5f} | val L1: {val_loss:.5f} | val PSNR: {val_psnr:.2f}")

        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(
                model,
                optimizer,
                epoch,
                os.path.join(CHECKPOINT_DIR, "sr_resnet_lite_best.pth"),
                extra={"val_loss": val_loss, "val_psnr": val_psnr}
            )

    return model


# Start with a short sanity training run.
# Increase epochs later.
sr_model = train_sr_model(sr_model, sr_train_loader, sr_val_loader, epochs=3, lr=1e-4)

## 9. Visualize SR predictions

In [ ]:
def visualize_sr_predictions(model, loader, n=2):
    model.eval()
    x, y, names = next(iter(loader))
    x = x.to(DEVICE)
    y = y.to(DEVICE)

    with torch.no_grad():
        pred = model(x)

    x = x.cpu()
    y = y.cpu()
    pred = pred.cpu()

    for i in range(min(n, x.size(0))):
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))

        ax[0].imshow(x[i, 0], cmap="gray")
        ax[0].set_title("Input TIR 200m\n256×256")
        ax[0].axis("off")

        ax[1].imshow(pred[i, 0], cmap="gray")
        ax[1].set_title("Predicted TIR 100m\n512×512")
        ax[1].axis("off")

        ax[2].imshow(y[i, 0], cmap="gray")
        ax[2].set_title("Target TIR 100m\n512×512")
        ax[2].axis("off")

        plt.suptitle(names[i])
        plt.tight_layout()
        plt.show()

visualize_sr_predictions(sr_model, sr_val_loader, n=2)

## 10. Colorization baseline model

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNetColorizer(nn.Module):
    '''
    Simple U-Net for TIR -> RGB colorization.

    Input:
      [B,1,256,256]

    Output:
      [B,3,256,256]
    '''
    def __init__(self):
        super().__init__()

        self.enc1 = ConvBlock(1, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.enc4 = ConvBlock(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = ConvBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        self.out = nn.Conv2d(32, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.out(d1))


color_model = UNetColorizer().to(DEVICE)

dummy = torch.randn(2, 1, 256, 256).to(DEVICE)
with torch.no_grad():
    out = color_model(dummy)
print("Color output shape:", out.shape)

## 11. Train colorization model

In [ ]:
def train_color_model(model, train_loader, val_loader, epochs=5, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for x, y, _ in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            pred = model(x)
            loss = criterion(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        val_psnr = 0.0

        with torch.no_grad():
            for x, y, _ in val_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)

                pred = model(x)
                loss = criterion(pred, y)

                val_loss += loss.item() * x.size(0)
                val_psnr += tensor_psnr(pred, y) * x.size(0)

        val_loss /= len(val_loader.dataset)
        val_psnr /= len(val_loader.dataset)

        print(f"Epoch {epoch:03d} | train L1: {train_loss:.5f} | val L1: {val_loss:.5f} | val PSNR: {val_psnr:.2f}")

        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(
                model,
                optimizer,
                epoch,
                os.path.join(CHECKPOINT_DIR, "unet_colorizer_best.pth"),
                extra={"val_loss": val_loss, "val_psnr": val_psnr}
            )

    return model


# Start with a short sanity training run.
# Increase epochs later.
color_model = train_color_model(color_model, color_train_loader, color_val_loader, epochs=3, lr=1e-4)

## 12. Visualize colorization predictions

In [ ]:
def visualize_color_predictions(model, loader, n=2):
    model.eval()
    x, y, names = next(iter(loader))
    x = x.to(DEVICE)
    y = y.to(DEVICE)

    with torch.no_grad():
        pred = model(x)

    x = x.cpu()
    y = y.cpu()
    pred = pred.cpu()

    for i in range(min(n, x.size(0))):
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))

        ax[0].imshow(x[i, 0], cmap="gray")
        ax[0].set_title("Input TIR 100m\n256×256")
        ax[0].axis("off")

        ax[1].imshow(np.clip(pred[i].permute(1, 2, 0).numpy(), 0, 1))
        ax[1].set_title("Predicted RGB")
        ax[1].axis("off")

        ax[2].imshow(np.clip(y[i].permute(1, 2, 0).numpy(), 0, 1))
        ax[2].set_title("Target RGB")
        ax[2].axis("off")

        plt.suptitle(names[i])
        plt.tight_layout()
        plt.show()

visualize_color_predictions(color_model, color_val_loader, n=2)

## 13. Evaluation on test sets

In [ ]:
def evaluate_sr(model, loader):
    model.eval()
    criterion = nn.L1Loss()
    total_loss = 0.0
    total_psnr = 0.0

    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            pred = model(x)
            loss = criterion(pred, y)
            total_loss += loss.item() * x.size(0)
            total_psnr += tensor_psnr(pred, y) * x.size(0)

    total_loss /= len(loader.dataset)
    total_psnr /= len(loader.dataset)

    return total_loss, total_psnr


def evaluate_color(model, loader):
    model.eval()
    criterion = nn.L1Loss()
    total_loss = 0.0
    total_psnr = 0.0

    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            pred = model(x)
            loss = criterion(pred, y)
            total_loss += loss.item() * x.size(0)
            total_psnr += tensor_psnr(pred, y) * x.size(0)

    total_loss /= len(loader.dataset)
    total_psnr /= len(loader.dataset)

    return total_loss, total_psnr


sr_test_loader = DataLoader(sr_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
color_test_loader = DataLoader(color_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

sr_loss, sr_psnr = evaluate_sr(sr_model, sr_test_loader)
color_loss, color_psnr = evaluate_color(color_model, color_test_loader)

print(f"SR Test L1: {sr_loss:.5f} | PSNR: {sr_psnr:.2f}")
print(f"Color Test L1: {color_loss:.5f} | PSNR: {color_psnr:.2f}")

## 14. End-to-end inference idea

The final challenge pipeline should eventually work like this:

```text
TIR 200m input
    ↓
SR model
    ↓
Predicted TIR 100m
    ↓
Colorization model
    ↓
Predicted RGB 100m
```

Important shape issue:

- SR output is `512×512`.
- Colorization model currently expects `256×256`.

For patch-based inference, use sliding windows. For quick testing, crop or tile the SR output into `256×256` windows before sending to colorization.

Final output TIFF requirement from the challenge:

```text
output/model_outputs/
  tir_superresolved_100m/
  colorized_tir_100m/
```

Colorized TIFF channel order should be:

```text
Layer 1 = Blue
Layer 2 = Green
Layer 3 = Red
```

Our in-memory RGB is `R,G,B`, so convert to `B,G,R` before final GeoTIFF saving.

## 15. GitHub project structure suggestion

In [ ]:
# Suggested repo structure:
#
# IR-colorization-BAH2026/
#   notebooks/
#     01_dataset_preparation.ipynb
#     02_model_training.ipynb
#   src/
#     datasets.py
#     models_sr.py
#     models_colorization.py
#     train_sr.py
#     train_colorization.py
#     inference.py
#     utils.py
#   docs/
#     TIR_SR_Colorization_Dataset_Documentation.md
#   configs/
#     sr_baseline.yaml
#     colorization_baseline.yaml
#   .gitignore
#   README.md
#
# Do not push:
#   *.npy
#   *.tif
#   *.zip
#   *.pth
#   /dataset_current_repo_format/
#   /checkpoints/
#
# Push only code, notebooks, docs, and small CSV metadata if needed.
print("GitHub structure note loaded.")

## 16. Minimal .gitignore

In [ ]:
gitignore_text = '''
# Python
__pycache__/
*.pyc
.ipynb_checkpoints/

# Data
*.npy
*.tif
*.tiff
*.zip
dataset/
dataset_current_repo_format/
landsat_india_200/
processed_100m_scenes/

# Model outputs
checkpoints/
*.pth
*.pt
runs/
wandb/

# OS
.DS_Store
'''

print(gitignore_text)